# Atividade 6 - Planejamento
## Enunciado:

---
### Problema 1
Um fiscal de agência de seguro avalia os valores de Prêmios cobrado por uma seguradora em 6 cidades. As cidades possuem duas características:
- **A = Tamanho** (Pequena, Média, Grande)  
- **B = Região** (Norte, Sul)

| Tamanho/Região | Norte | Sul |
|----------------|-------|-----|
| Pequena        | 140   | 100 |
| Média          | 210   | 180 |
| Grande         | 220   | 220 |

- **item a:** Faça análise completa dos dados do problema.  
- **item b:** Use o teste de aditividade de Tukey para verificar a interação.  
- **item c:** Caso positivo, use o modelo de regressão para elaborar a análise.

---

## Problema 1 — Análise Completa

### Estrutura Experimental

Este é um experimento **fatorial 3 × 2 sem replicação**, em que:
- **Fator A** = Tamanho da cidade (Pequena, Média, Grande) — $a = 3$ níveis  
- **Fator B** = Região (Norte, Sul) — $b = 2$ níveis  
- **Variável resposta**: Prêmio cobrado (em unidades monetárias)

Como há apenas **uma observação por cela**, não é possível estimar a interação diretamente a partir da ANOVA clássica — o termo de erro e a interação são confundidos.  
Por isso, o modelo assumido inicialmente é o **modelo aditivo**:

$$Y_{ij} = \mu + \alpha_i + \beta_j + \varepsilon_{ij}$$

onde:
- $Y_{ij}$: prêmio observado no nível $i$ do Tamanho e $j$ da Região  
- $\mu$: média geral  
- $\alpha_i$: efeito do nível $i$ do Fator A (Tamanho), com $\sum \alpha_i = 0$  
- $\beta_j$: efeito do nível $j$ do Fator B (Região), com $\sum \beta_j = 0$  
- $\varepsilon_{ij} \sim N(0,\sigma^2)$: erro aleatório

### Hipóteses

**Para o Fator A (Tamanho):**

$$H_0: \alpha_1 = \alpha_2 = \alpha_3 = 0$$
$$H_1: \text{Pelo menos um } \alpha_i \neq 0$$

**Para o Fator B (Região):**

$$H_0: \beta_1 = \beta_2 = 0$$
$$H_1: \text{Pelo menos um } \beta_j \neq 0$$

### Item a — Análise Completa

#### 1.1 Entrada dos dados e estatística descritiva

In [ ]:
# Entrada dos dados
premio <- c(140, 100,
            210, 180,
            220, 220)

tamanho <- factor(rep(c("Pequena", "Media", "Grande"), each = 2),
                  levels = c("Pequena", "Media", "Grande"))
regiao  <- factor(rep(c("Norte", "Sul"), times = 3),
                  levels = c("Norte", "Sul"))

dados1 <- data.frame(Tamanho = tamanho, Regiao = regiao, Premio = premio)
print(dados1)

In [ ]:
library(dplyr)

# Estatísticas por Tamanho (Fator A)
dados1 %>%
  group_by(Tamanho) %>%
  summarise(Media = mean(Premio), DP = sd(Premio), n = n())

In [ ]:
# Estatísticas por Região (Fator B)
dados1 %>%
  group_by(Regiao) %>%
  summarise(Media = mean(Premio), DP = sd(Premio), n = n())

In [ ]:
# Tabela de médias cruzadas
tapply(dados1$Premio, list(dados1$Tamanho, dados1$Regiao), mean)

#### 1.2 Gráficos exploratórios

O gráfico de perfis de médias é especialmente útil para inspecionar visualmente a presença de interação: se as linhas forem paralelas, o modelo aditivo é adequado; caso contrário, há indício de interação.

In [ ]:
par(mfrow = c(1, 2))

# Gráfico de interação: Tamanho × Região
interaction.plot(dados1$Tamanho, dados1$Regiao, dados1$Premio,
                 type = "b", col = c("steelblue", "tomato"), lwd = 2, pch = 19,
                 xlab = "Tamanho", ylab = "Prêmio médio",
                 trace.label = "Região",
                 main = "Perfis de médias: Tamanho × Região")

# Gráfico de interação invertido: Região × Tamanho
interaction.plot(dados1$Regiao, dados1$Tamanho, dados1$Premio,
                 type = "b", col = c("steelblue", "tomato", "darkgreen"),
                 lwd = 2, pch = 19,
                 xlab = "Região", ylab = "Prêmio médio",
                 trace.label = "Tamanho",
                 main = "Perfis de médias: Região × Tamanho")

par(mfrow = c(1, 1))

Os perfis de médias apresentam linhas visualmente próximas do paralelismo, sugerindo ausência de interação — mas isso será testado formalmente pelo teste de Tukey (item b).

---

#### 1.3 Ajuste do Modelo Aditivo e ANOVA

In [ ]:
# Ajuste do modelo aditivo (sem interação — experimento sem replicação)
modelo1 <- aov(Premio ~ Tamanho + Regiao, data = dados1)
summary(modelo1)

#### 1.4 Verificação dos Pressupostos

Com apenas 6 observações e 2 graus de liberdade residual, os testes têm pouca potência, mas são realizados por completude.

In [ ]:
residuos1  <- residuals(modelo1)
ajustados1 <- fitted(modelo1)

# Teste de Shapiro-Wilk (normalidade)
shapiro.test(residuos1)

In [ ]:
par(mfrow = c(1, 2))

# Q-Q Plot
qqnorm(residuos1, main = "Q-Q Plot dos Resíduos")
qqline(residuos1, col = "red", lwd = 2)

# Resíduos vs Ajustados
plot(ajustados1, residuos1,
     xlab = "Valores Ajustados", ylab = "Resíduos",
     main = "Resíduos vs Ajustados", pch = 19)
abline(h = 0, col = "red", lwd = 2)

par(mfrow = c(1, 1))

#### 1.5 Interpretação da ANOVA

Com base na tabela ANOVA:

- **Fator A (Tamanho):** Se o valor-p for pequeno (< 0,05), rejeita-se $H_0$, concluindo que o tamanho da cidade influencia significativamente o prêmio cobrado.  
- **Fator B (Região):** Analogamente, verifica-se o efeito da região.

Como não há replicação, a interação $AB$ não é estimada diretamente — ela seria confundida com o erro. O teste formal de interação é o de Tukey, realizado no item b.

---

### Item b — Teste de Aditividade de Tukey

O **teste de aditividade de Tukey (1949)** avalia formalmente se a interação $AB$ é nula no modelo sem replicação. A ideia é decompor a SS do resíduo em uma componente atribuível à interação (1 g.l.) e o restante.

A estatística é baseada em:

$$SS_{\text{interação}} = \frac{\left(\sum_{i,j} y_{ij}\, \hat{\alpha}_i\, \hat{\beta}_j\right)^2}{SS_A \cdot SS_B / SS_T^*}$$

onde $\hat{\alpha}_i = \bar{y}_{i.} - \bar{y}_{..}$ e $\hat{\beta}_j = \bar{y}_{.j} - \bar{y}_{..}$.

As hipóteses são:

$$H_0: \text{não há interação (modelo aditivo é adequado)}$$
$$H_1: \text{existe interação não-aditiva}$$

In [ ]:
# ── Teste de Aditividade de Tukey (implementação manual) ──────────────────────

Y <- matrix(dados1$Premio, nrow = 3, ncol = 2,
            dimnames = list(c("Pequena","Media","Grande"), c("Norte","Sul")))

a <- nrow(Y)   # 3 níveis de Tamanho
b <- ncol(Y)   # 2 níveis de Região

media_geral  <- mean(Y)
media_linha  <- rowMeans(Y)         # médias por Tamanho
media_coluna <- colMeans(Y)         # médias por Região

# efeitos estimados
alpha_hat <- media_linha  - media_geral
beta_hat  <- media_coluna - media_geral

cat("Média geral:", media_geral, "\n")
cat("Efeitos de Tamanho (alpha_hat):\n"); print(alpha_hat)
cat("Efeitos de Região (beta_hat):\n");  print(beta_hat)

In [ ]:
# Numerador da estatística de Tukey
num <- sum(outer(alpha_hat, beta_hat) * Y)^2

# SQA e SQB (usadas no denominador)
SQA <- b * sum(alpha_hat^2)
SQB <- a * sum(beta_hat^2)

# SS_interação (1 g.l.)
SS_inter <- num / (SQA * SQB)

# SS_resíduo total e SS_residuo puro (após retirar interação)
SS_res_total <- sum((Y - outer(media_linha, rep(1,b)) -
                         outer(rep(1,a), media_coluna) +
                         media_geral)^2)

SS_res_puro <- SS_res_total - SS_inter

cat("\nSS_interação (1 g.l.) =", round(SS_inter, 4), "\n")
cat("SS_resíduo total      =", round(SS_res_total, 4), "\n")
cat("SS_resíduo puro       =", round(SS_res_puro, 4), "\n")

# Estatística F
gl_res_puro <- (a - 1) * (b - 1) - 1
F_tukey <- (SS_inter / 1) / (SS_res_puro / gl_res_puro)
p_tukey  <- pf(F_tukey, df1 = 1, df2 = gl_res_puro, lower.tail = FALSE)

cat("\nF_Tukey =", round(F_tukey, 4),
    " | g.l. = 1 e", gl_res_puro,
    " | valor-p =", round(p_tukey, 4), "\n")

#### Interpretação do Teste de Aditividade de Tukey

- Se **valor-p > 0,05**: não há evidência de interação — o modelo aditivo é adequado e o item c não é necessário.  
- Se **valor-p ≤ 0,05**: há interação significativa — o modelo aditivo não é adequado e prossegue-se ao item c.

---

### Item c — Modelo de Regressão (caso a interação seja detectada)

Se o teste de Tukey detectar interação, utiliza-se a transformação sugerida por Tukey para estabilizar a não-aditividade. A transformação é da forma $Y^{\lambda}$, em que o expoente ótimo é estimado por:

$$\lambda = 1 - \frac{\hat{\mu} \cdot \hat{\gamma}}{SS_{\text{interação}} / SS_{\text{resíduo}}}$$

Uma abordagem prática é ajustar o modelo de regressão com a variável resposta transformada pela Box-Cox, ou incluir explicitamente um termo de interação baseado no produto $\hat{\alpha}_i \cdot \hat{\beta}_j$.

Abaixo, o modelo de regressão via dummies é ajustado incluindo o termo de interação de Tukey:

In [ ]:
# ── Modelo de regressão com term de interação de Tukey ───────────────────────

# Criar o termo de interação de Tukey: produto dos efeitos estimados
dados1$alpha_hat <- alpha_hat[as.character(dados1$Tamanho)]
dados1$beta_hat  <- beta_hat[as.character(dados1$Regiao)]
dados1$tukey_term <- dados1$alpha_hat * dados1$beta_hat

print(dados1)

In [ ]:
# Dummies manuais (referência: Pequena e Norte)
dados1$Tamanho_Media  <- as.numeric(dados1$Tamanho == "Media")
dados1$Tamanho_Grande <- as.numeric(dados1$Tamanho == "Grande")
dados1$Regiao_Sul     <- as.numeric(dados1$Regiao == "Sul")

# Modelo de regressão completo com interação de Tukey
mod_reg1 <- lm(Premio ~ Tamanho_Media + Tamanho_Grande + Regiao_Sul + tukey_term,
               data = dados1)

summary(mod_reg1)

In [ ]:
# ANOVA do modelo de regressão
anova(mod_reg1)

In [ ]:
# Diagnóstico do modelo de regressão
res_reg1 <- residuals(mod_reg1)
fit_reg1 <- fitted(mod_reg1)

par(mfrow = c(1, 2))

qqnorm(res_reg1, main = "Q-Q Plot — Modelo de Regressão")
qqline(res_reg1, col = "red", lwd = 2)

plot(fit_reg1, res_reg1,
     xlab = "Valores Ajustados", ylab = "Resíduos",
     main = "Resíduos vs Ajustados", pch = 19)
abline(h = 0, col = "red", lwd = 2)

par(mfrow = c(1, 1))

#### Interpretação do Modelo de Regressão

- O coeficiente do **termo de interação de Tukey** (`tukey_term`) indica a magnitude da não-aditividade.  
- Se ele for significativo, confirma-se a interação e o modelo aditivo simples deve ser substituído.  
- Os demais coeficientes estimam os efeitos principais de Tamanho e Região, controlada a interação.

---
---

---
### Problema 2
Um hormônio de crescimento sintético é administrado em crianças com deficiência em produção desse hormônio. A variável resposta é a **diferença entre as taxas de crescimento** (antes e depois do uso). O pesquisador tem interesse em avaliar o efeito do **sexo** e do **grau de desenvolvimento ósseo** (Severo, Moderado, Leve). Em cada grupo, 3 crianças foram aleatoriamente alocadas, porém **4 famílias desistiram** do experimento (dados desbalanceados).

| Sexo \ Grau | Severo | Moderado | Leve |
|-------------|--------|----------|------|
| Masculino | 1.4, 2.4, 2.2 | 2.1, 1.7 | 0.7, 1.1 |
| Feminino  | 2.4            | 2.5, 1.8, 2.0 | 0.5, 0.9, 1.3 |

**item:** Faça análise completa desses dados.

---

## Problema 2 — Análise Completa

### Estrutura Experimental

Este é um experimento **fatorial 2 × 3 com replicação desigual** (dados desbalanceados), com:
- **Fator A** = Sexo (Masculino, Feminino) — $a = 2$ níveis  
- **Fator B** = Grau de desenvolvimento ósseo (Severo, Moderado, Leve) — $b = 3$ níveis  
- **Variável resposta**: Diferença na taxa de crescimento

O modelo com interação é:

$$Y_{ijk} = \mu + \alpha_i + \beta_j + (\alpha\beta)_{ij} + \varepsilon_{ijk}$$

onde:
- $Y_{ijk}$: $k$-ésima observação no nível $i$ de Sexo e nível $j$ de Grau  
- $\mu$: média geral  
- $\alpha_i$: efeito do $i$-ésimo nível de Sexo  
- $\beta_j$: efeito do $j$-ésimo nível de Grau  
- $(\alpha\beta)_{ij}$: efeito da interação  
- $\varepsilon_{ijk} \sim N(0, \sigma^2)$: erro aleatório

### Hipóteses

**Para a Interação AB:**

$$H_0: (\alpha\beta)_{ij} = 0 \quad \forall i,j$$
$$H_1: \text{Pelo menos uma interação} \neq 0$$

**Para o Fator A (Sexo):**

$$H_0: \alpha_1 = \alpha_2 = 0 \qquad H_1: \text{Pelo menos um } \alpha_i \neq 0$$

**Para o Fator B (Grau):**

$$H_0: \beta_1 = \beta_2 = \beta_3 = 0 \qquad H_1: \text{Pelo menos um } \beta_j \neq 0$$

### 2.1 Entrada dos dados e estatística descritiva

Como o experimento é desbalanceado (número diferente de observações por cela), usaremos a ANOVA **Tipo III** (soma de quadrados ajustada), implementada via `car::Anova()`, que é robusta ao desbalanceamento.

In [ ]:
# Entrada dos dados desbalanceados
taxa <- c(
  # Masculino - Severo  (3 obs)
  1.4, 2.4, 2.2,
  # Masculino - Moderado (2 obs)
  2.1, 1.7,
  # Masculino - Leve     (2 obs)
  0.7, 1.1,
  # Feminino  - Severo   (1 obs)
  2.4,
  # Feminino  - Moderado (3 obs)
  2.5, 1.8, 2.0,
  # Feminino  - Leve     (3 obs)
  0.5, 0.9, 1.3
)

sexo <- factor(c(
  rep("Masculino", 3+2+2),
  rep("Feminino",  1+3+3)
))

grau <- factor(c(
  rep("Severo",   3), rep("Moderado", 2), rep("Leve", 2),   # Masculino
  rep("Severo",   1), rep("Moderado", 3), rep("Leve", 3)    # Feminino
), levels = c("Severo", "Moderado", "Leve"))

dados2 <- data.frame(Sexo = sexo, Grau = grau, Taxa = taxa)
print(dados2)

In [ ]:
library(dplyr)

# Número de observações por cela
cat("── Contagem por cela (n_ij):\n")
print(table(dados2$Sexo, dados2$Grau))

# Médias por cela
cat("\n── Médias por cela:\n")
print(tapply(dados2$Taxa, list(dados2$Sexo, dados2$Grau), mean))

# Desvios-padrão por cela
cat("\n── Desvios-padrão por cela:\n")
print(tapply(dados2$Taxa, list(dados2$Sexo, dados2$Grau), sd))

In [ ]:
# Estatísticas marginais por Sexo
cat("── Médias marginais por Sexo:\n")
dados2 %>%
  group_by(Sexo) %>%
  summarise(Media = mean(Taxa), DP = sd(Taxa), n = n()) %>%
  print()

# Estatísticas marginais por Grau
cat("\n── Médias marginais por Grau:\n")
dados2 %>%
  group_by(Grau) %>%
  summarise(Media = mean(Taxa), DP = sd(Taxa), n = n()) %>%
  print()

#### 2.2 Gráficos exploratórios

In [ ]:
par(mfrow = c(1, 2))

# Perfis de médias: Sexo × Grau
medias_cela <- tapply(dados2$Taxa, list(dados2$Sexo, dados2$Grau), mean)

interaction.plot(dados2$Grau, dados2$Sexo, dados2$Taxa,
                 type = "b",
                 col  = c("steelblue", "tomato"),
                 lwd  = 2, pch = 19,
                 xlab = "Grau de Desenvolvimento Ósseo",
                 ylab = "Taxa de crescimento média",
                 trace.label = "Sexo",
                 main = "Perfis de médias: Grau × Sexo")

# Boxplot por Grau, separado por Sexo
boxplot(Taxa ~ Grau, data = dados2,
        col   = c("steelblue", "tomato", "darkgreen"),
        xlab  = "Grau de Desenvolvimento Ósseo",
        ylab  = "Taxa de crescimento",
        main  = "Distribuição por Grau")

par(mfrow = c(1, 1))

#### 2.3 Ajuste do Modelo Fatorial com Interação (ANOVA Tipo III)

Como o experimento é **desbalanceado**, a ANOVA padrão do R (`aov` + `summary`) calcula somas de quadrados do **Tipo I (sequencial)**, cujo resultado depende da ordem dos termos. Para dados desbalanceados, o correto é usar **Tipo III (marginal)**, disponível em `car::Anova()`.

In [ ]:
library(car)

# Ajuste do modelo com interação via lm (necessário para Anova Tipo III)
# Contraste soma-zero para interpretação correta dos efeitos principais com interação
options(contrasts = c("contr.sum", "contr.poly"))

modelo2 <- lm(Taxa ~ Sexo * Grau, data = dados2)

# ANOVA Tipo III
cat("── ANOVA Tipo III:\n")
Anova(modelo2, type = "III")

#### 2.4 Verificação dos Pressupostos

In [ ]:
residuos2  <- residuals(modelo2)
ajustados2 <- fitted(modelo2)

# Normalidade
sw2 <- shapiro.test(residuos2)
cat("Shapiro-Wilk: W =", round(sw2$statistic, 4),
    "| valor-p =", round(sw2$p.value, 4), "\n")

In [ ]:
# Homocedasticidade — Levene e Bartlett (por combinação Sexo:Grau)
dados2$grupo <- interaction(dados2$Sexo, dados2$Grau)
leveneTest(Taxa ~ grupo, data = dados2)

In [ ]:
par(mfrow = c(1, 2))

qqnorm(residuos2, main = "Q-Q Plot dos Resíduos")
qqline(residuos2, col = "red", lwd = 2)

plot(ajustados2, residuos2,
     xlab = "Valores Ajustados", ylab = "Resíduos",
     main = "Resíduos vs Ajustados", pch = 19)
abline(h = 0, col = "red", lwd = 2)

par(mfrow = c(1, 1))

#### 2.5 Comparações Múltiplas

Se a interação $AB$ **não for significativa**, analisa-se os efeitos principais com comparações múltiplas (Tukey).  
Se a interação **for significativa**, as comparações são feitas dentro de cada nível dos fatores (desdobramento).

Abaixo, aplicamos o teste de Tukey de forma geral:

In [ ]:
# Reajustar com aov para uso do TukeyHSD
modelo2_aov <- aov(Taxa ~ Sexo * Grau, data = dados2)

# Tukey HSD para o Grau (fator com 3 níveis)
cat("── Tukey HSD — Grau:\n")
TukeyHSD(modelo2_aov, "Grau")

In [ ]:
# Tukey HSD para Sexo
cat("── Tukey HSD — Sexo:\n")
TukeyHSD(modelo2_aov, "Sexo")

In [ ]:
# Gráfico dos intervalos de confiança de Tukey
par(mfrow = c(1, 2))
plot(TukeyHSD(modelo2_aov, "Grau"), las = 1, main = "Tukey — Grau")
plot(TukeyHSD(modelo2_aov, "Sexo"), las = 1, main = "Tukey — Sexo")
par(mfrow = c(1, 1))

#### 2.6 Interpretação Final

Com base na ANOVA Tipo III e nos testes de comparações múltiplas:

1. **Interação Sexo × Grau**: avalia se o efeito do Grau de desenvolvimento ósseo sobre a taxa de crescimento é diferente entre meninos e meninas.  
2. **Efeito principal do Grau**: espera-se que crianças com grau Severo apresentem maior taxa de crescimento (maior resposta ao hormônio), enquanto as com grau Leve apresentem menor resposta.  
3. **Efeito principal do Sexo**: verifica-se se há diferença entre sexos na resposta média ao hormônio, independentemente do grau.  
4. Os **pressupostos** (normalidade e homocedasticidade) devem ser verificados; com $n$ pequeno, o teste de Shapiro-Wilk tem baixa potência, e os gráficos de diagnóstico são fundamentais.

> **Nota sobre o desbalanceamento**: a desistência de 4 famílias resultou em uma cela com apenas 1 observação (Feminino–Severo), o que elimina o grau de liberdade do resíduo dessa cela. Esse é o principal impacto do desbalanceamento neste experimento: reduz a precisão das estimativas e inviabiliza testes de normalidade e homocedasticidade por cela.

---
---

## Problema 2 — Modelo de Regressão com Variáveis Dummy

### Por que usar regressão?

A ANOVA e o modelo de regressão linear são **matematicamente equivalentes** para experimentos fatoriais. A regressão torna explícita a estrutura dos parâmetros estimados ao criar variáveis indicadoras (dummies) para os níveis dos fatores. Isso é especialmente útil para:

- Visualizar diretamente os **coeficientes** $\hat{\mu}$, $\hat{\alpha}_i$, $\hat{\beta}_j$ e $\widehat{(\alpha\beta)}_{ij}$;
- Realizar **testes F parciais** (Tipo III) sem precisar de funções auxiliares;
- Estender facilmente para covariáveis ou estruturas mais complexas.

### Modelo

$$Y_{ijk} = \mu + \alpha_i X_i + \beta_j Z_j + \gamma_{ij} X_i Z_j + \varepsilon_{ijk}$$

Com as codificações *dummy* (referência: Masculino e Severo):

| Variável | Significado |
|----------|-------------|
| $X_1$    | 1 se Feminino, 0 c.c. |
| $Z_1$    | 1 se Moderado, 0 c.c. |
| $Z_2$    | 1 se Leve, 0 c.c.     |
| $X_1 Z_1$ | interação Feminino × Moderado |
| $X_1 Z_2$ | interação Feminino × Leve     |

*Set to zero*: $\alpha_{\text{Masculino}} = 0$, $\beta_{\text{Severo}} = 0$ (categorias de referência).

O vetor de parâmetros é:
$$\boldsymbol{\beta} = (\mu,\; \alpha_1,\; \beta_1,\; \beta_2,\; \gamma_{11},\; \gamma_{12})^\top$$

### 2.7 Construção das variáveis dummy

In [ ]:
# Os dados já foram criados anteriormente; recriamos aqui por completude
taxa <- c(1.4, 2.4, 2.2,   # Masc - Severo
          2.1, 1.7,         # Masc - Moderado
          0.7, 1.1,         # Masc - Leve
          2.4,              # Fem  - Severo
          2.5, 1.8, 2.0,   # Fem  - Moderado
          0.5, 0.9, 1.3)   # Fem  - Leve

sexo <- factor(c(rep("Masculino", 7), rep("Feminino", 7)))
grau <- factor(c(rep("Severo",3), rep("Moderado",2), rep("Leve",2),
                 rep("Severo",1), rep("Moderado",3), rep("Leve",3)),
               levels = c("Severo","Moderado","Leve"))

dados2 <- data.frame(Sexo = sexo, Grau = grau, Taxa = taxa)

# ── Dummies manuais (referência: Masculino e Severo) ──────────────────────────
dados2$Fem       <- as.numeric(dados2$Sexo == "Feminino")
dados2$Moderado  <- as.numeric(dados2$Grau == "Moderado")
dados2$Leve      <- as.numeric(dados2$Grau == "Leve")

# Termos de interação
dados2$Fem_Mod  <- dados2$Fem * dados2$Moderado
dados2$Fem_Leve <- dados2$Fem * dados2$Leve

print(dados2)

### 2.8 Ajuste do modelo de regressão

O modelo completo com interação é ajustado via `lm()`. Note que o R com `contr.treatment` (default) já usa a codificação de referência — ao criar as dummies manualmente temos controle total sobre qual categoria é a referência.

In [ ]:
# Modelo completo com dummies manuais
mod_reg2 <- lm(Taxa ~ Fem + Moderado + Leve + Fem_Mod + Fem_Leve,
               data = dados2)

summary(mod_reg2)

#### Interpretação dos coeficientes

| Coeficiente | Parâmetro | Significado |
|-------------|-----------|-------------|
| `(Intercept)` | $\mu$ | Média de Masculino–Severo |
| `Fem` | $\alpha_1$ | Diferença Feminino − Masculino no grau Severo |
| `Moderado` | $\beta_1$ | Diferença Moderado − Severo para Masculino |
| `Leve` | $\beta_2$ | Diferença Leve − Severo para Masculino |
| `Fem_Mod` | $\gamma_{11}$ | Quanto a diferença Moderado − Severo muda para Feminino |
| `Fem_Leve` | $\gamma_{12}$ | Quanto a diferença Leve − Severo muda para Feminino |

As médias de cela preditas pelo modelo são:

$$\hat{Y}_{ij} = \hat{\mu} + \hat{\alpha}_i + \hat{\beta}_j + \hat{\gamma}_{ij}$$

Com o modelo de regressão **saturado** (6 parâmetros = 6 celas), os valores preditos coincidem exatamente com as médias observadas por cela.

### 2.9 Médias preditas por cela

In [ ]:
# Tabela de novas observações para predição (uma por cela)
grid_pred <- data.frame(
  Sexo     = factor(c("Masculino","Masculino","Masculino",
                       "Feminino","Feminino","Feminino"),
                    levels = c("Masculino","Feminino")),
  Grau     = factor(c("Severo","Moderado","Leve",
                       "Severo","Moderado","Leve"),
                    levels = c("Severo","Moderado","Leve")),
  Fem      = c(0,0,0,1,1,1),
  Moderado = c(0,1,0,0,1,0),
  Leve     = c(0,0,1,0,0,1),
  Fem_Mod  = c(0,0,0,0,1,0),
  Fem_Leve = c(0,0,0,0,0,1)
)

grid_pred$Media_predita <- predict(mod_reg2, newdata = grid_pred)

# Médias observadas por cela (para comparação)
grid_pred$Media_obs <- tapply(dados2$Taxa,
                              list(dados2$Sexo, dados2$Grau),
                              mean)[cbind(as.character(grid_pred$Sexo),
                                          as.character(grid_pred$Grau))]

print(grid_pred[, c("Sexo","Grau","Media_obs","Media_predita")])

### 2.10 Tabela ANOVA do modelo de regressão

A tabela ANOVA do `lm` pelo método Tipo I é sequencial e depende da ordem dos termos. Para dados desbalanceados, usamos novamente o **Tipo III** via `car::Anova()`.

In [ ]:
library(car)

cat("── ANOVA Tipo I (sequencial) — lm:\n")
anova(mod_reg2)

cat("\n── ANOVA Tipo III (marginal) — car::Anova:\n")
Anova(mod_reg2, type = "III")

#### Comparação com a ANOVA fatorial

As SS e os valores-p da ANOVA Tipo III devem coincidir com os obtidos na seção 2.3, confirmando a equivalência entre as duas abordagens.

Os testes F do modelo de regressão têm estrutura:

$$F = \frac{SS_{\text{parcial}} / \text{g.l.}}{QME}$$

onde $QME = SS_{\text{resíduo}} / (n - p)$, com $n = 14$ observações e $p = 6$ parâmetros, logo $n - p = 8$ graus de liberdade residuais.

### 2.11 Testes F parciais — comparação de modelos encaixados

In [ ]:
# Modelo sem interação (efeitos principais apenas)
mod_principal <- lm(Taxa ~ Fem + Moderado + Leve, data = dados2)

# Modelo sem Sexo (apenas Grau + interação não faz sentido; teste marginal de Sexo)
mod_sem_sexo <- lm(Taxa ~ Moderado + Leve, data = dados2)

# Modelo sem Grau
mod_sem_grau <- lm(Taxa ~ Fem, data = dados2)

# Teste F: modelo com interação vs. sem interação
cat("── Teste F: interação Sexo × Grau (modelo completo vs. só efeitos principais)\n")
anova(mod_principal, mod_reg2)

cat("\n── Teste F: efeito de Sexo (marginal)\n")
anova(mod_sem_sexo, mod_reg2)

cat("\n── Teste F: efeito de Grau (marginal)\n")
anova(mod_sem_grau, mod_reg2)

### 2.12 Diagnóstico dos resíduos do modelo de regressão

In [ ]:
res_reg2  <- residuals(mod_reg2)
fit_reg2  <- fitted(mod_reg2)

# Shapiro-Wilk
sw_reg2 <- shapiro.test(res_reg2)
cat("Shapiro-Wilk: W =", round(sw_reg2$statistic, 4),
    "| valor-p =", round(sw_reg2$p.value, 4), "\n")

par(mfrow = c(1, 2))

qqnorm(res_reg2, main = "Q-Q Plot — Regressão")
qqline(res_reg2, col = "red", lwd = 2)

plot(fit_reg2, res_reg2,
     xlab = "Valores Ajustados", ylab = "Resíduos",
     main = "Resíduos vs Ajustados", pch = 19, col = "steelblue")
abline(h = 0, col = "red", lwd = 2)

par(mfrow = c(1, 1))

### 2.13 Alavancagem e influência

Com $n = 14$ e $p = 6$ parâmetros, o limiar de alavancagem é $2p/n \approx 0{,}857$.  
A cela **Feminino–Severo** tem apenas 1 observação, o que tende a produzir alto $h_{ii}$.

In [ ]:
X   <- model.matrix(mod_reg2)
H   <- X %*% solve(t(X) %*% X) %*% t(X)
hii <- diag(H)

p_reg <- length(coef(mod_reg2))
n_reg <- nrow(dados2)
limiar_h <- 2 * p_reg / n_reg

cook_reg <- cooks.distance(mod_reg2)

diag_reg <- data.frame(
  obs        = 1:n_reg,
  Sexo       = dados2$Sexo,
  Grau       = dados2$Grau,
  Taxa       = dados2$Taxa,
  Ajustado   = round(fit_reg2, 3),
  Residuo    = round(res_reg2, 3),
  hii        = round(hii, 3),
  Cook       = round(cook_reg, 3),
  Alta_alav  = hii > limiar_h
)

print(diag_reg)

In [ ]:
par(mfrow = c(1, 2))

plot(hii, type = "h",
     col  = ifelse(hii > limiar_h, "red", "steelblue"),
     ylim = c(0, 1),
     main = "Alavancagem (h_ii)",
     ylab = "h_ii", xlab = "Observação")
abline(h = limiar_h, col = "red", lty = 2)
text(which(hii > limiar_h), hii[hii > limiar_h] + 0.03,
     labels = which(hii > limiar_h), col = "red", cex = 0.8)

plot(cook_reg, type = "h",
     col  = ifelse(cook_reg > 1, "red", "steelblue"),
     ylim = c(0, max(cook_reg) * 1.2),
     main = "Distância de Cook",
     ylab = "Cook's D", xlab = "Observação")
abline(h = 1, col = "red", lty = 2)

par(mfrow = c(1, 1))

### 2.14 Comparação entre ANOVA e Regressão

| Aspecto | ANOVA (`car::Anova`) | Regressão (`lm` + dummies) |
|---------|----------------------|---------------------------|
| Testes F (Tipo III) | ✔ idênticos | ✔ idênticos |
| Coeficientes interpretáveis | ✗ (apenas médias) | ✔ (efeitos e interações explícitos) |
| Médias preditas por cela | via `model.tables` | via `predict()` |
| Alavancagem / Cook | não direto | ✔ direto (`hatvalues`, `cooks.distance`) |
| Extensão para covariáveis | difícil | ✔ natural (ANCOVA) |

**Conclusão**: as duas abordagens produzem os **mesmos testes e as mesmas conclusões**. A regressão com dummies é mais flexível e expõe diretamente a estrutura paramétrica do modelo, sendo preferível quando se deseja interpretar efeitos, avaliar influência de observações individuais ou extender o modelo.